# Minutes of Board meetings processor

## Configuration

In [38]:
WORK_DIR = "minutes_of_board_meetings"
INPUT_URL = "https://www.uni-corvinus.hu/fooldal/egyetemunkrol/maecenas-universitatis-corvini-alapitvany/kozzeteteli-lista/"

## URL crawler

In [33]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [9]:
# 1. Download HTML

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/151.0.0.0 Safari/537.36"
    )
}

response = requests.get(INPUT_URL, headers=headers, timeout=30)
response.raise_for_status()

# Use the encoding detected by requests
response.encoding = response.apparent_encoding

soup = BeautifulSoup(response.text, "html.parser")

In [10]:
# 2. Find the requested div
container = soup.find("div", id="ac-532362104")

if container is None:
    raise ValueError("Could not find div with id='ac-532362104'.")

In [11]:
# 3. Find accordion sections
records = []

# Look through all elements that could represent accordion titles
for title in container.find_all(["h2", "h3", "h4", "h5", "button", "div"]):

    title_text = title.get_text(" ", strip=True)

    # Find a four-digit year, e.g. 2024, 2025, 2026
    year_match = re.search(r"\b(20\d{2})\b", title_text)

    if not year_match:
        continue

    year = int(year_match.group(1))

    # ---------------------------------------------------------------
    # Find links belonging to this accordion section.
    #
    # We first look in the parent element and then progressively
    # larger parents if necessary.
    # ---------------------------------------------------------------

    parent = title.parent

    links = []

    for _ in range(5):
        if parent is None:
            break

        links = parent.find_all("a", href=True)

        if links:
            break

        parent = parent.parent

    # ---------------------------------------------------------------
    # Extract filenames
    # ---------------------------------------------------------------

    for link in links:

        filename = link.get_text(" ", strip=True)

        href = link.get("href")

        if not filename or not href:
            continue

        # Ignore navigation/non-file links
        if not re.search(
            r"\.(pdf|docx?|xlsx?|pptx?|zip|csv|txt)(?:$|\?)",
            href,
            flags=re.IGNORECASE,
        ):
            continue

        records.append(
            {
                "year": year,
                "filename": filename,
                "url": href,
            }
        )


In [12]:
records[:5]

[{'year': 2019,
  'filename': 'kuratóriumi jegyzőkönyv 12.17',
  'url': '/downloads/dfpg.1k01p9n/kuratoriumi-jegyzokonyv-12.17.pdf'},
 {'year': 2019,
  'filename': 'Jegyzőkönyv 20191106',
  'url': '/downloads/dfpe.nbep14/jegyzokonyv-20191106.pdf'},
 {'year': 2019,
  'filename': 'Jegyzőkönyv 20190627',
  'url': '/downloads/dfpd.1wmy07l/jegyzokonyv-20190627.pdf'},
 {'year': 2019,
  'filename': 'kuratóriumi jegyzőkönyv 11.27',
  'url': '/downloads/dfpf.10lprog/kuratoriumi-jegyzokonyv-11.27.pdf'},
 {'year': 2019,
  'filename': 'jegyzőkönyv 20190604',
  'url': '/downloads/dfpc.930n2h/jegyzokonyv-20190604.pdf'}]

In [15]:
# 4. Convert

df_files = (
    pd.DataFrame(records)
    .drop_duplicates(subset=["year", "filename", "url"])
    .reset_index(drop=True)
)

In [16]:
df_files

,year,filename,url
0,2019,kuratóriumi jegyzőkönyv 12.17,/downloads/dfpg.1k01p9n/kuratoriumi-jegyzokony...
1,2019,Jegyzőkönyv 20191106,/downloads/dfpe.nbep14/jegyzokonyv-20191106.pdf
2,2019,Jegyzőkönyv 20190627,/downloads/dfpd.1wmy07l/jegyzokonyv-20190627.pdf
3,2019,kuratóriumi jegyzőkönyv 11.27,/downloads/dfpf.10lprog/kuratoriumi-jegyzokony...
4,2019,jegyzőkönyv 20190604,/downloads/dfpc.930n2h/jegyzokonyv-20190604.pdf
...,...,...,...
763,2026,Jegyzőkönyv 06.23,/downloads/dce2.k9ub9l/jegyzokonyv-06.23.pdf
764,2026,jegyzőkönyv 06.26,/downloads/dce3.183pyhd/jegyzokonyv-06.26.pdf
765,2026,jegyzőkönyv 0130,/downloads/dce4.1pfq49r/jegyzokonyv-0130.pdf
766,2026,jegyzőkönyv 0428,/downloads/dce5.2xg953/jegyzokonyv-0428.pdf


## Downloading files

In [1]:
import os

import requests

from urllib.parse import urljoin, urlparse, unquote

In [5]:
WORK_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = (
    "https://www.uni-corvinus.hu/fooldal/egyetemunkrol/"
    "maecenas-universitatis-corvini-alapitvany/kozzeteteli-lista/"
)

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/151.0.0.0 Safari/537.36"
    )
}

In [ ]:
session = requests.Session()
session.headers.update(headers)

for index, row in df.iterrows():

    year = str(row["year"])
    filename = str(row["filename"])
    url = urljoin(BASE_URL, str(row["url"]))

    # Make filename safe for the local filesystem
    filename = filename.strip()
    filename = "".join(
        c if c not in '<>:"/\\|?*' else "_"
        for c in filename
    )

    output_filename = f"{year}_{filename}"
    output_path = WORK_DIR / output_filename

    print(f"Downloading: {url}")
    print(f"       -> {output_path}")

    try:
        response = session.get(url, timeout=60)
        response.raise_for_status()

        with open(output_path, "wb") as f:
            f.write(response.content)

        print(f"OK: {output_path}")

    except requests.RequestException as e:
        print(f"ERROR: {url}")
        print(f"       {e}")


print("\nDownload complete.")

## OCRing the PDFs

In [2]:
from pathlib import Path
import re

import fitz  # PyMuPDF
import pandas as pd
import pytesseract
from PIL import Image

In [ ]:
# OCR language
# Make sure Hungarian Tesseract data is installed:
# https://github.com/tesseract-ocr/tessdata

OCR_LANGUAGE = "hun"
OCR_DPI = 300

In [4]:
def extract_pdf_text(pdf_path: Path) -> str:
    """
    Extract text from a PDF.

    First tries normal PDF text extraction.
    If the PDF is scanned/image-based, falls back to OCR.
    """

    print(f"Processing: {pdf_path.name}")

    with fitz.open(pdf_path) as doc:

        # -----------------------------------------------------------
        # 1. Try native PDF text extraction
        # -----------------------------------------------------------

        native_text = "\n\n".join(
            page.get_text("text")
            for page in doc
        ).strip()

        # If enough text was extracted, use it
        if len(native_text) > 100:
            print("  -> Native text extraction")
            return native_text

        # -----------------------------------------------------------
        # 2. OCR fallback
        # -----------------------------------------------------------

        print("  -> Scanned PDF detected, using OCR")

        ocr_pages = []

        zoom = OCR_DPI / 72
        matrix = fitz.Matrix(zoom, zoom)

        for page_number, page in enumerate(doc, start=1):

            print(f"     OCR page {page_number}/{len(doc)}")

            # Render page to image
            pix = page.get_pixmap(
                matrix=matrix,
                alpha=False
            )

            image = Image.frombytes(
                "RGB",
                [pix.width, pix.height],
                pix.samples
            )

            # OCR
            text = pytesseract.image_to_string(
                image,
                lang=OCR_LANGUAGE,
                config="--psm 3"
            )

            ocr_pages.append(text.strip())

        return "\n\n".join(ocr_pages)


https://digi.bib.uni-mannheim.de/tesseract/

In [ ]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Users\zoltan.balogh\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"

print(pytesseract.get_languages())

In [ ]:
if not WORK_DIR.exists():
    raise FileNotFoundError(
        f"Output directory does not exist: {WORK_DIR}"
    )

pdf_files = sorted(OUTPUT_DIR.glob("*"))

print(f"Found {len(pdf_files)} PDF files\n")


records = []

for pdf_path in pdf_files:

    try:
        text = extract_pdf_text(pdf_path)

        # Extract year from filename, e.g.
        # 2020_some_document.pdf -> 2020
        year_match = re.match(
            r"^(\d{4})_",
            pdf_path.name
        )

        year = (
            int(year_match.group(1))
            if year_match
            else None
        )

        records.append({
            "year": year,
            "filename": pdf_path.name,
            "text": text,
        })

        print(f"  Extracted {len(text):,} characters\n")

    except Exception as e:
        print(
            f"ERROR processing {pdf_path.name}: {e}\n"
        )

In [ ]:
from Helper import Log

In [ ]:
df = pd.DataFrame(records)

In [ ]:
with Log.save_file("output2/docs_ocr.csv") as filename:
    df.to_csv(filename, index=False)

In [ ]:
df_claude = pd.read_csv("output/docs_ocr_202608180010.csv")
df_deepl = pd.read_csv("output2/docs_ocr_202609191347.csv")

In [ ]:
df_claude[:5]

In [ ]:
df_deepl[:5]

In [ ]:
required = {"filename"}
for name, frame in {"df_claude": df_claude, "df_deepl": df_deepl}.items():
    missing = required - set(frame.columns)
    if missing:
        raise KeyError(f"{name} is missing columns: {missing}")

claude_filenames = set(df_claude["filename"].dropna().astype(str))
deepl_filenames = set(df_deepl["filename"].dropna().astype(str))

print(f"Claude records: {len(df_claude):,}")
print(f"DeepL records: {len(df_deepl):,}")
print(f"Matching filenames: {len(claude_filenames & deepl_filenames):,}")
print(f"Only in Claude: {len(claude_filenames - deepl_filenames):,}")
print(f"Only in DeepL: {len(deepl_filenames - claude_filenames):,}")

print("\nDuplicate filenames:")
print("Claude:", df_claude["filename"].duplicated().sum())
print("DeepL:", df_deepl["filename"].duplicated().sum())

df_all = df_claude.merge(
    df_deepl,
    on="filename",
    how="outer",
    suffixes=("_claude", "_deepl"),
    indicator=True,
)

df_all["_merge"].value_counts()

In [ ]:
CHUNK_SIZE = 100

for start in range(0, len(df), CHUNK_SIZE):
    chunk = df.iloc[start:start + CHUNK_SIZE]
    part_number = start // CHUNK_SIZE + 1

    with Log.save_file(f"output2/docs_ocr_part{part_number}.json") as filename:
        chunk.to_json(
            filename,
            orient="records",
            force_ascii=False,
            indent=2,
        )

## Translation

### OVH

In [ ]:
import yaml
from pathlib import Path

import pandas as pd

from tqdm.auto import tqdm

from langchain.chat_models import init_chat_model

In [ ]:
def load_yaml_config(config_path: str | Path) -> dict:
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"Config file not found: {path}")

    with path.open("r", encoding="utf-8") as handle:
        config = yaml.safe_load(handle) or {}

    for root_name, root_value in config.items():
        if key := root_value.get("key"):
            if key is True:
                key_path = path.parent / f"{root_name}.key"
                if key_path.exists():
                    root_value["key"] = key_path.read_text(encoding="utf-8").strip()

    return config

In [ ]:
DEFAULT_LLM_TIMEOUT_SECONDS = 120.0

def create_response_model(config_path: str = "configs/ovh.yaml", root_node: str = "ovh_gpt") -> object:
    config = load_yaml_config(config_path)
    model_config = config[root_node]

    return init_chat_model(
        model=model_config["model"],
        model_provider=model_config["provider"],
        base_url=model_config["base_url"],
        temperature=model_config["temperature"],
        reasoning_effort=model_config["reasoning_effort"],
        api_key=model_config["key"],
        timeout=model_config.get("timeout", DEFAULT_LLM_TIMEOUT_SECONDS),
    )

In [ ]:
response_model = create_response_model(config_path="C:\\Users\\zoltan.balogh\\Documents\\qa_agent\\configs\\ovh.yaml", root_node="qwen35_9b_medium")
# qwen35_9b_medium
# qwen36_27b_high
# gpt_oss_20b_high

In [ ]:
df = pd.read_csv("output\\docs_ocr_202608180010.csv")

In [ ]:
translated_texts = []

for _, row in tqdm(df[:5].iterrows(), total=len(df[:5]), desc="Translating"):
    text = row["text"]

    if pd.isna(text) or not str(text).strip():
        translated_texts.append(text)
        continue

    response = response_model.invoke(
        [
            (
                "system",
                "You are a precise Hungarian-to-English translator. "
                "Return only the English translation and preserve the "
                "document's structure and meaning.",
            ),
            (
                "human",
                f"Translate the following Hungarian text into English:\n\n{text}",
            ),
        ]
    )
    translated_texts.append(response.content.strip())

df["text_english"] = translated_texts

In [ ]:
print(df.loc[0, 'text'])

In [ ]:
len(df.loc[0, 'text'])

In [ ]:
response = response_model.invoke(
        [
            (
                "system",
                "You are a precise Hungarian-to-English translator. "
                "Return only the English translation and preserve the "
                "document's structure and meaning.",
            ),
            (
                "human",
                "Translate the following Hungarian text into English:\n\n{:}".format(
                    df.loc[0, 'text']
                ),
            )
        ]
    )

In [ ]:
response.content.strip()

### HuggingFace
https://huggingface.co/google/translategemma-4b-it

In [ ]:
import pandas as pd

from Helper import Log

In [ ]:
df = pd.read_csv("output\\docs_ocr_202608180010.csv")

In [ ]:
df[:5]

In [ ]:
import os

import torch
from huggingface_hub import get_token, login
from tqdm.auto import tqdm
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_NAME = "google/translategemma-4b-it"
SOURCE_LANG_CODE = "hu"
TARGET_LANG_CODE = "en"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_DTYPE = torch.bfloat16 if DEVICE.type == "cuda" else torch.float32
MAX_SOURCE_TOKENS = 1500
MAX_NEW_TOKENS = 512

HF_TOKEN = os.getenv("HF_TOKEN") or get_token()
if HF_TOKEN is None:
    login(add_to_git_credential=False)
    HF_TOKEN = get_token()

if HF_TOKEN is None:
    raise RuntimeError(
        "Hugging Face authentication failed. Set HF_TOKEN or complete login()."
    )

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    torch_dtype=MODEL_DTYPE,
).to(DEVICE)
model.eval()


def translate_hungarian_text(text: object) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    token_ids = processor.tokenizer.encode(text, add_special_tokens=False)
    translated_chunks: list[str] = []

    for start in range(0, len(token_ids), MAX_SOURCE_TOKENS):
        chunk = processor.tokenizer.decode(
            token_ids[start : start + MAX_SOURCE_TOKENS],
            skip_special_tokens=True,
        )
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "source_lang_code": SOURCE_LANG_CODE,
                        "target_lang_code": TARGET_LANG_CODE,
                        "text": chunk,
                    }
                ],
            }
        ]
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(DEVICE)
        input_len = inputs["input_ids"].shape[-1]

        with torch.inference_mode():
            generation = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
            )

        translated_chunks.append(
            processor.decode(
                generation[0][input_len:],
                skip_special_tokens=True,
            ).strip()
        )

    return "\n".join(
        chunk for chunk in translated_chunks if chunk
    )

In [ ]:
translate_hungarian_text(df.loc[0, 'text'])

In [ ]:
for i in tqdm(range(len(df)), desc="Translating"):
    text = df.loc[i, "text"]

    if pd.isna(text) or not str(text).strip():
        df.loc[i, "text_english"] = text
        continue

    df.loc[i, "text_english"] = translate_hungarian_text(text)

    if (i + 1) % 50 == 0:
        with Log.save_file(f"output/docs_ocr_202608180010_part{(i + 1) // 50}.csv") as filename:
            df.to_csv(filename, sep="\t", index=False)

In [ ]:
df[:5]

In [ ]:
print(df.loc[1, "text_english"])

In [ ]:
with Log.save_file("output/docs_ocr_202608180010_translated.csv") as filename:
    df.to_csv(filename, sep="\t", index=False)

### Claude

Set `ANTHROPIC_API_KEY` in the notebook kernel's environment before running the client cell. You can optionally set `CLAUDE_MODEL`, `CLAUDE_MAX_TOKENS`, and `CLAUDE_TIMEOUT_SECONDS`; adjust the translation range in the configuration cell.

In [ ]:
import os
from pathlib import Path
from typing import Any

import anthropic
import pandas as pd

from Helper import Log
from tqdm.auto import tqdm

In [ ]:
if not os.environ.get("ANTHROPIC_API_KEY"):
    api_key = Path("./anthropic.key").read_text(encoding="utf-8").strip()
    if not api_key:
        raise ValueError("An Anthropic API key is required.")

    os.environ["ANTHROPIC_API_KEY"] = api_key
    del api_key
else:
    print("ANTHROPIC_API_KEY is already set.")

In [ ]:
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5")
CLAUDE_MAX_TOKENS = int(os.getenv("CLAUDE_MAX_TOKENS", "8192"))
CLAUDE_TIMEOUT_SECONDS = float(os.getenv("CLAUDE_TIMEOUT_SECONDS", "300"))

TRANSLATION_START = 552
TRANSLATION_STOP = None
CHECKPOINT_EVERY = 50

CLAUDE_SYSTEM_PROMPT = (
    "You are a precise Hungarian-to-English translator. "
    "Return only the English translation. "
    "Preserve the source document's structure, paragraph breaks, lists, "
    "headings, numbering, names, dates, and meaning. "
    "Do not add explanations or commentary."
)


def create_claude_client(api_key: str | None = None) -> anthropic.Anthropic:
    resolved_api_key = api_key or os.getenv("ANTHROPIC_API_KEY")
    if not resolved_api_key:
        raise RuntimeError(
            "Set the ANTHROPIC_API_KEY environment variable before creating the client."
        )

    return anthropic.Anthropic(
        api_key=resolved_api_key,
        timeout=CLAUDE_TIMEOUT_SECONDS,
        max_retries=3,
    )

In [ ]:
def translate_hungarian_text(
    client: anthropic.Anthropic,
    text: Any,
    *,
    model: str = CLAUDE_MODEL,
) -> Any:
    if text is None or pd.isna(text) or not str(text).strip():
        return text

    response = client.messages.create(
        model=model,
        max_tokens=CLAUDE_MAX_TOKENS,
        system=CLAUDE_SYSTEM_PROMPT,
        messages=[
            {
                "role": "user",
                "content": (
                    "Translate the following Hungarian text into English.\n\n"
                    f"{str(text)}"
                ),
            }
        ],
    )

    translated_text = "".join(
        block.text
        for block in response.content
        if getattr(block, "type", None) == "text"
    ).strip()

    if not translated_text:
        raise ValueError("Claude returned no translated text.")

    return translated_text

In [ ]:
claude_client = None

if os.getenv("ANTHROPIC_API_KEY"):
    claude_client = create_claude_client()
else:
    print("Set ANTHROPIC_API_KEY and rerun this cell before translating.")

In [ ]:
INPUT_PATH = "output/docs_ocr_202608180010.csv"

df = pd.read_csv(INPUT_PATH)

if "text" not in df.columns:
    raise KeyError(f"Expected a 'text' column in {INPUT_PATH}.")

if "text_english" not in df.columns:
    df["text_english"] = pd.Series(index=df.index, dtype="object")

In [ ]:
if claude_client is None:
    raise RuntimeError(
        "Claude client is not configured. Set ANTHROPIC_API_KEY and rerun the client cell."
    )

start = max(0, TRANSLATION_START)
stop = len(df) if TRANSLATION_STOP is None else min(TRANSLATION_STOP, len(df))

if start > stop:
    raise ValueError("TRANSLATION_START must be less than or equal to TRANSLATION_STOP.")

for position in tqdm(
    range(start, stop),
    total=stop - start,
    desc="Translating with Claude",
):
    index = df.index[position]
    text = df.at[index, "text"]

    try:
        df.at[index, "text_english"] = translate_hungarian_text(
            claude_client,
            text,
            model=CLAUDE_MODEL,
        )
    except Exception as exc:
        raise RuntimeError(
            f"Claude translation failed for dataframe position {position} "
            f"(index {index})."
        ) from exc

    if CHECKPOINT_EVERY and (position + 1) % CHECKPOINT_EVERY == 0:
        with Log.save_file(
            f"output/docs_ocr_202608180010_claude_part{(position + 1) // CHECKPOINT_EVERY}.csv"
        ) as filename:
            df.to_csv(filename, index=False)

In [ ]:
OUTPUT_PATH = "output/docs_ocr_202608180010_claude_translated.csv"

with Log.save_file(OUTPUT_PATH) as filename:
    df.to_csv(filename, index=False)

In [ ]:
print(df.loc[550, "text"])

In [ ]:
print(translate_hungarian_text(
    claude_client,
    df.loc[551, "text"],
    model=CLAUDE_MODEL,
))

## Joining the translations

In [36]:
from Helper import Log

In [39]:
json_files = [f for f in Path(WORK_DIR + "/json").glob("*.json")]

if not json_files:
    raise FileNotFoundError("No JSON files found in the data2 folder.")

df_deepl_content = pd.concat(
    [pd.read_json(path) for path in json_files],
    ignore_index=True,
)

print(f"Loaded {len(json_files)} files and {len(df_deepl_content)} records.")
df_deepl_content.head()

Loaded 8 files and 752 records.


,filename,text,year
0,2019_register-20200624,MINUTES OF THE BOARD OF TRUSTEES MEETING\n\nwh...,2019
1,2019_Minutes 0130,MINUTES\n\nOf the joint meeting of the Board o...,2019
2,2019_minutes 03.09,MINUTES\n\nOF THE MEETING OF THE BOARD OF TRUS...,2019
3,2019_Minutes 0428,MINUTES\n\nOF THE MEETING OF THE BOARD OF TRUS...,2019
4,2019_minutes 0527,MINUTES\n\nOf the joint meeting of the Board o...,2019


In [40]:
with Log.save_file(WORK_DIR + "/minutes_of_board_meetings.csv") as filename:
    df_deepl_content.to_csv(filename)

File saved to `minutes_of_board_meetings\minutes_of_board_meetings_202609231043.csv` (6.6MB)
